# Training data chip creation
something descriptive about 300x300 dataset

In [1]:
# !rm -rf ~/.local/lib/python*

In [2]:
# !pip install rioxarray geopandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached rioxarray-0.22.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached xarray-2026.4.0-py3-none-any.whl.metadata (12 kB)
  Using cached pyogrio-0.12.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.9 kB)
Using cached rioxarray-0.22.0-py3-none-any.whl (72 kB)
Using cached pyogrio-0.12.1-cp312-cp312-manylinux_2_28_x86_64.whl (32.5 MB)
Using cached xarray-2026.4.0-py3-none-any.whl (1.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [rioxarray]/3 [rioxarray]


In [3]:
import json
import logging
import os
import sys
import time
import shutil
from glob import glob
from pathlib import Path
from contextlib import redirect_stdout, nullcontext
from functools import partial
from collections import Counter
from io import StringIO
import multiprocessing as mp
from multiprocessing import Pool, cpu_count, get_logger

import geopandas as gpd
import numpy as np
import rasterio
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.crs import CRS
from tqdm import tqdm
import subprocess

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

In [4]:
# !rm -rf ./lfm

In [5]:
repo_dir = "lfm"

try:
    if not os.path.exists(repo_dir):
        result = subprocess.run(
            ["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"Successfully cloned repository to {repo_dir}/")
    else:
        result = subprocess.run(
            ["git", "-C", repo_dir, "pull"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"Successfully pulled latest changes in {repo_dir}/")
except (subprocess.CalledProcessError, FileNotFoundError) as e:
    print(e)
    print("Git subprocess call failed (known issue in Jupyter notebooks)")
    print("\nPlease run this command in your terminal:")
    print(f"  git clone https://github.com/nasa-nccs-hpda/{repo_dir}")

repoPath = Path(os.getcwd()) / repo_dir
sys.path.append(str(repoPath.parent))

Successfully cloned repository to lfm/


In [6]:
result2 = subprocess.run(
    ["git", "-C", repo_dir, "checkout", "develop"],
    check=True,
    capture_output=True,
    text=True
)

In [7]:
from lfm.model.Pipeline import Pipeline
from lfm.model.chip_making.chip_utils import extract_product_id, run_pipeline_for_sample, get_worker_logger
from lfm.model.chip_making.chip_constants import PROJECT_DIR, GPKG_PATH, TILE_DB_PATH, ZOOM_LEVEL

In [8]:
logger = get_worker_logger()

# User configuration

In [9]:
output_dir = PROJECT_DIR / "model_inputs/notebook_outputs"
output_dir.mkdir(exist_ok=True)

# Example run: single train sample

## 1. Load training geometries into GeoDataFrame

In [10]:
train_gdf_path = GPKG_PATH
train_gdf = gpd.read_file(GPKG_PATH)
print(f"Loaded train gdf with {len(train_gdf)} entries.")

Loaded train gdf with 674 entries.


## 2. Run datacube pipeline for single sample

In [11]:
entry = train_gdf.iloc[0].to_dict()
entry

{'location': 'M1096558039CE_r7650_c750_input.tif',
 'geometry': <POLYGON ((-11.389 -5.164, -10.396 -5.168, -10.4 -6.158, -11.395 -6.152, -11...>}

In [12]:
train_fn = entry['location']
product_id = extract_product_id(train_fn)
print(f"Product ID: {product_id}")

# Save to datacube subdirectory
train_fn_no_ext = train_fn.replace('.tif', '')
datacube_base_dir = output_dir / "datacubes"
datacube_dir = datacube_base_dir / train_fn_no_ext
datacube_dir.mkdir(exist_ok=True, parents=True)

Product ID: M1096558039CE


In [17]:
print("Running pipeline...")

pipeline_start = time.time()

cube_files, pipeline_status = run_pipeline_for_sample(
    train_fn=train_fn,
    product_id=product_id,
    geom_bounds=entry['geometry'].bounds,  # (ulLon, lrLat, lrLon, ulLat)
    datacube_dir=datacube_dir,
    TILE_DB_PATH=TILE_DB_PATH,
    logger=logger
)
pipeline_end = time.time() - pipeline_start

if not cube_files:
    print("WARNING: no datacubes created.")
if pipeline_status != "success":
    print(f"WARNING: Pipeline status: {pipeline_status}")
else:
    print(f"Pipeline successfully ran for train sample. Number of files created: {len(cube_files)}")
    print(f"Pipeline ran in {pipeline_end} seconds.")

Running pipeline...


TypeError: run_pipeline_for_sample() missing 1 required positional argument: 'zoom_level'